# 04d（US）样本外（Walk-forward）：按年检验 `stem` 效应稳定性

目的：把“样本内显著”推进到“样本外是否稳定”。

输入：
- `data/clean_us/market_ganzhi_{ts_code}.csv.gz`

输出（写入 `data/clean_us/robustness/`）：
- `walk_forward_{ts_code}_stem.csv`：按年 walk-forward 的稳定性（Spearman + `丙` 的 OOS 差异）
- `walk_forward_{ts_code}_stem_oos_effects.csv`：OOS 年度效应长表（raw；每年×每 stem）
- `walk_forward_{ts_code}_stem_oos_effects_controls.csv`：OOS 年度效应长表（controls-only residual）
- `walk_forward_{ts_code}_stem_oos_effects_controls_sweep.csv`：训练窗敏感性（`train_years` sweep）


In [ ]:
from __future__ import annotations

import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from patsy import build_design_matrices
from scipy import stats

warnings.filterwarnings('ignore')

# statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf

# plots
import matplotlib.pyplot as plt
import seaborn as sns



from matplotlib import font_manager, rcParams


def set_chinese_font() -> None:
    # Best-effort Chinese font setup for Matplotlib.
    # - Tries common Linux CJK fonts
    # - On WSL, also tries registering Windows fonts under /mnt/c/Windows/Fonts

    candidates = [
        'Noto Sans CJK SC',
        'Noto Sans CJK TC',
        'Noto Sans CJK JP',
        'Source Han Sans SC',
        'WenQuanYi Micro Hei',
        'Microsoft YaHei',
        'SimHei',
        'SimSun',
        'Arial Unicode MS',
    ]

    # WSL fallback: register Windows fonts if available
    try:
        from pathlib import Path as _Path

        win_dir = _Path('/mnt/c/Windows/Fonts')
        win_files = [
            win_dir / 'msyh.ttc',
            win_dir / 'msyhbd.ttc',
            win_dir / 'simhei.ttf',
            win_dir / 'simsun.ttc',
            win_dir / 'arialuni.ttf',
        ]

        extra_names: list[str] = []
        addfont = getattr(font_manager.fontManager, 'addfont', None)
        for fp in win_files:
            try:
                if fp.is_file():
                    if callable(addfont):
                        addfont(str(fp))
                    extra_names.append(font_manager.FontProperties(fname=str(fp)).get_name())
            except Exception:
                continue

        candidates = [n for n in extra_names if n] + candidates
    except Exception:
        pass

    for name in candidates:
        try:
            font_manager.findfont(name, fallback_to_default=False)
            rcParams['font.sans-serif'] = [name]
            rcParams['axes.unicode_minus'] = False
            print(f'Using Chinese font: {name}')
            return
        except Exception:
            continue

    rcParams['axes.unicode_minus'] = False
    print(
        'Warning: no Chinese font found; Chinese labels may not render. '
        'If you are on WSL, ensure Windows fonts are accessible under /mnt/c/Windows/Fonts; '
        'or install Noto CJK fonts in Linux (e.g., fonts-noto-cjk).'
    )


set_chinese_font()

def find_project_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()

    for candidate in [here] + list(here.parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'data').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate

    for candidate in here.glob('*'):
        if candidate.is_dir() and (candidate / 'AGENTS.md').is_file() and (candidate / 'data').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate

    return here


ROOT = find_project_root()
print('PROJECT_ROOT =', ROOT)

CLEAN_DIR = ROOT / 'data/clean_us'
ROBUST_DIR = CLEAN_DIR / 'robustness'
ROBUST_DIR.mkdir(parents=True, exist_ok=True)

# =====================
# Config (edit as needed)
# =====================
TS_CODES = ['spx', 'ndq', 'ndx', 'dji']

# Control-variable models to run
GROUP_COLS = ['stem', 'branch']
RUN_GANZHI_DAY_MODELS = True

# Permutation test (raw series; global)
RANDOM_SEED = 20260213
N_PERM = 1000

# Permutation on controls-only residuals (more conservative)
N_PERM_RESID = 2000

# HAC / Newey-West (time-series dependence)
HAC_MAXLAGS = 5
HAC_MAXLAGS_SWEEP = [0, 1, 3, 5, 10]

# Block bootstrap (controls-only residuals)
N_BOOT = 1000
BOOT_BLOCK_LEN = 10

# Walk-forward
TRAIN_YEARS = 5
TRAIN_YEARS_SWEEP = [3, 5, 7]

STEMS_ORDER = list('甲乙丙丁戊己庚辛壬癸')
BRANCHES_ORDER = list('子丑寅卯辰巳午未申酉戌亥')

# Phase 2: Li-Chun year element interaction (day_group × year_element)
RUN_PHASE2_INTERACTIONS = True
PHASE2_DAY_GROUPS = ['stem', 'branch', 'ganzhi_day']
YEAR_ELEMENT_ORDER = ['木', '火', '土', '金', '水']
PHASE2_P_THRESHOLD = 0.10
PHASE2_GATE_Q = 0.10
PHASE2_GATE_MIN_SHARE = 0.75


def fdr_bh(pvals: np.ndarray) -> np.ndarray:
    # Benjamini-Hochberg FDR q-values for a 1D array.
    p = np.asarray(pvals, dtype=float)
    n = p.size
    order = np.argsort(p)
    q = np.empty(n, dtype=float)
    prev = 1.0

    for rank in range(n - 1, -1, -1):
        i = order[rank]
        val = p[i] * n / (rank + 1)
        prev = min(prev, val)
        q[i] = prev

    return np.clip(q, 0.0, 1.0)


def load_market_ganzhi(ts_code: str) -> pd.DataFrame:
    path = CLEAN_DIR / f'market_ganzhi_{ts_code}.csv.gz'
    if not path.exists():
        raise FileNotFoundError(f'Missing: {path}. Run notebooks_US 01-03 first.')
    df = pd.read_csv(path, compression='gzip', dtype={'trade_date': str})
    if df.empty:
        raise ValueError(f'Empty data: {path}')
    return df


def add_calendar_controls(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    dt = pd.to_datetime(out['trade_date'], format='%Y%m%d')
    out['date'] = dt
    out['weekday'] = dt.dt.weekday.astype(int)
    out['month'] = dt.dt.month.astype(int)
    out['year'] = dt.dt.year.astype(int)
    return out


def set_category_orders(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if 'stem' in out.columns:
        out['stem'] = pd.Categorical(out['stem'], categories=STEMS_ORDER, ordered=True)
    if 'branch' in out.columns:
        out['branch'] = pd.Categorical(out['branch'], categories=BRANCHES_ORDER, ordered=True)
    return out




## 1) Walk-forward：稳定性统计（Spearman + 丙的 OOS 差异）


In [ ]:
def spearman_corr(a: np.ndarray, b: np.ndarray) -> float:
    ra = pd.Series(a).rank().to_numpy(dtype=float)
    rb = pd.Series(b).rank().to_numpy(dtype=float)
    if np.std(ra) == 0 or np.std(rb) == 0:
        return np.nan
    return float(np.corrcoef(ra, rb)[0, 1])


def walk_forward_stem(ts_code: str, train_years: int) -> pd.DataFrame:
    df = load_market_ganzhi(ts_code)
    df = add_calendar_controls(df)
    df = set_category_orders(df)
    df = df.dropna(subset=['ret_1d', 'up', 'stem'])

    years = sorted(df['year'].unique().tolist())
    min_y, max_y = int(min(years)), int(max(years))

    records = []
    for oos_year in range(min_y + train_years, max_y + 1):
        train = df[(df['year'] >= oos_year - train_years) & (df['year'] <= oos_year - 1)]
        oos = df[df['year'] == oos_year]

        if len(train) < 200 or len(oos) < 50:
            continue

        train_g = train.groupby('stem').agg(mean_ret=('ret_1d', 'mean'), p_up=('up', 'mean'))
        oos_g = oos.groupby('stem').agg(mean_ret=('ret_1d', 'mean'), p_up=('up', 'mean'))

        idx = [s for s in STEMS_ORDER if s in train_g.index and s in oos_g.index]
        if len(idx) < 6:
            continue

        train_mean = train_g.loc[idx, 'mean_ret'].to_numpy()
        oos_mean = oos_g.loc[idx, 'mean_ret'].to_numpy()
        train_p = train_g.loc[idx, 'p_up'].to_numpy()
        oos_p = oos_g.loc[idx, 'p_up'].to_numpy()

        def _diff_bing(gdf: pd.DataFrame, col: str) -> float:
            if '丙' not in gdf.index:
                return np.nan
            return float(gdf.loc['丙', col] - gdf[col].mean())

        records.append(
            {
                'ts_code': ts_code,
                'oos_year': oos_year,
                'train_years': train_years,
                'n_train': int(len(train)),
                'n_oos': int(len(oos)),
                'spearman_mean_ret': spearman_corr(train_mean, oos_mean),
                'spearman_p_up': spearman_corr(train_p, oos_p),
                'train_bing_mean_ret_diff': _diff_bing(train_g, 'mean_ret'),
                'oos_bing_mean_ret_diff': _diff_bing(oos_g, 'mean_ret'),
            }
        )

    return pd.DataFrame.from_records(records)


wf_all = []
for ts_code in TS_CODES:
    wf = walk_forward_stem(ts_code, TRAIN_YEARS)
    out_path = ROBUST_DIR / f'walk_forward_{ts_code}_stem.csv'
    wf.to_csv(out_path, index=False)
    print('saved:', out_path, '| rows=', len(wf))
    wf_all.append(wf)

wf_df = pd.concat(wf_all, ignore_index=True)
wf_df.head()


# ------------------------------


## 2) Walk-forward：OOS 效应长表（raw）


In [ ]:
# Walk-forward: OOS effects for all stems (long table)
# ------------------------------

def walk_forward_stem_oos_effects(ts_code: str, train_years: int) -> pd.DataFrame:
    df = load_market_ganzhi(ts_code)
    df = add_calendar_controls(df)
    df = set_category_orders(df)
    df = df.dropna(subset=['ret_1d', 'stem'])

    years = sorted(df['year'].unique().tolist())
    min_y, max_y = int(min(years)), int(max(years))

    records = []
    for oos_year in range(min_y + int(train_years), max_y + 1):
        oos = df[df['year'] == oos_year]
        if len(oos) < 50:
            continue

        oos_g = oos.groupby('stem').agg(mean_ret=('ret_1d', 'mean'))
        idx = [s for s in STEMS_ORDER if s in oos_g.index]
        if len(idx) < 6:
            continue

        baseline = float(oos_g.loc[idx, 'mean_ret'].mean())
        for stem in idx:
            eff = float(oos_g.loc[stem, 'mean_ret'] - baseline)
            records.append({'ts_code': ts_code, 'oos_year': int(oos_year), 'stem': str(stem), 'oos_effect': eff})

    return pd.DataFrame.from_records(records)


for ts_code in TS_CODES:
    eff = walk_forward_stem_oos_effects(ts_code, TRAIN_YEARS)
    out_path = ROBUST_DIR / f'walk_forward_{ts_code}_stem_oos_effects.csv'
    eff.to_csv(out_path, index=False)
    print('saved:', out_path, '| rows=', len(eff))


# ------------------------------


## 3) Walk-forward：OOS 效应长表（controls-only residual）


In [ ]:
# Walk-forward (controls-only residual): OOS effects for all stems (long table)
# ------------------------------
# 目的：解释候选 stem（如 癸/丁）为何 Meta 显著但 OOS 不稳：
# 先回归掉 weekday/month（不含 year，避免把年份当作可泛化信息），再看 OOS 年度残差均值差。

def walk_forward_stem_oos_effects_controls(ts_code: str, train_years: int) -> pd.DataFrame:
    df = load_market_ganzhi(ts_code)
    df = add_calendar_controls(df)
    df = set_category_orders(df)
    df = df.dropna(subset=['ret_1d', 'weekday', 'month', 'stem']).copy()

    years = sorted(df['year'].unique().tolist())
    min_y, max_y = int(min(years)), int(max(years))

    records = []
    for oos_year in range(min_y + int(train_years), max_y + 1):
        train = df[(df['year'] >= oos_year - int(train_years)) & (df['year'] <= oos_year - 1)].copy()
        oos = df[df['year'] == oos_year].copy()

        if len(train) < 200 or len(oos) < 50:
            continue

        # controls-only model: no year dummies to avoid leakage / non-generalizable fit
        try:
            res = smf.ols(formula='ret_1d ~ C(weekday) + C(month)', data=train).fit()
        except Exception:
            continue

        oos['ret_hat_controls'] = res.predict(oos)
        oos['ret_resid_controls'] = oos['ret_1d'] - oos['ret_hat_controls']

        oos_g = oos.groupby('stem').agg(mean_resid=('ret_resid_controls', 'mean'))
        idx = [s for s in STEMS_ORDER if s in oos_g.index]
        if len(idx) < 6:
            continue

        baseline = float(oos_g.loc[idx, 'mean_resid'].mean())
        for stem in idx:
            eff = float(oos_g.loc[stem, 'mean_resid'] - baseline)
            records.append({'ts_code': ts_code, 'oos_year': int(oos_year), 'stem': str(stem), 'oos_effect': eff})

    return pd.DataFrame.from_records(records)


# Default window (TRAIN_YEARS)
for ts_code in TS_CODES:
    eff = walk_forward_stem_oos_effects_controls(ts_code, TRAIN_YEARS)
    out_path = ROBUST_DIR / f'walk_forward_{ts_code}_stem_oos_effects_controls.csv'
    eff.to_csv(out_path, index=False)
    print('saved:', out_path, '| rows=', len(eff))


# Window sweep (for sensitivity): save a combined long table with train_years column
for ts_code in TS_CODES:
    all_eff = []
    for ty in TRAIN_YEARS_SWEEP:
        eff = walk_forward_stem_oos_effects_controls(ts_code, ty)
        eff['train_years'] = int(ty)
        all_eff.append(eff)
    out = pd.concat(all_eff, ignore_index=True) if all_eff else pd.DataFrame(columns=['ts_code','oos_year','stem','oos_effect','train_years'])
    out_path = ROBUST_DIR / f'walk_forward_{ts_code}_stem_oos_effects_controls_sweep.csv'
    out.to_csv(out_path, index=False)
    print('saved:', out_path, '| rows=', len(out))


